# Polymarket Python Quickstart 2026 – Discover Markets, Analyze Odds & Trade

Welcome to the official Polymarket Python Quickstart guide for 2026! This notebook will walk you through everything you need to know to interact with Polymarket's prediction markets programmatically using Python.

## What is Polymarket?

Polymarket is a leading decentralized prediction market platform that allows users to trade on the outcome of events. Unlike traditional betting platforms, Polymarket uses blockchain technology to create transparent, censorship-resistant markets where users can buy and sell shares representing their beliefs about future outcomes.

## How Prediction Markets Work

Prediction markets operate on a simple principle: the wisdom of crowds. By allowing people to put real money behind their predictions, these markets aggregate information and often produce forecasts that outperform individual experts.

On Polymarket:
- Each market represents a question about a future event (e.g., "Will Candidate X win the election?")
- Markets typically have two or more outcomes (e.g., "Yes" or "No")
- Shares of each outcome trade between $0 and $1
- The current price of a share can be interpreted as the probability of that outcome occurring
- When the event resolves, shares of the correct outcome pay out $1 each, while shares of incorrect outcomes pay $0

This creates a powerful incentive mechanism where those with accurate information are rewarded for participating and sharing their knowledge through trading.

Let's get started with using Python to interact with these markets!

# ⚠️ Important Risk & Safety Warnings ⚠️

Before proceeding with this guide, please be aware of the following important considerations:

## Financial Risk

- **Real Money**: Polymarket involves trading with real cryptocurrency (USDC.e on Polygon). All trades, positions, and orders involve actual financial risk.
- **No Guarantees**: Past performance is not indicative of future results. Prediction markets can be volatile and unpredictable.
- **Start Small**: If you're new to Polymarket, begin with small amounts until you're comfortable with the platform and API.

## Technical Considerations

- **Polygon Gas Fees**: All transactions on Polymarket require Polygon (MATIC) for gas fees. Ensure your wallet has sufficient MATIC to cover transaction costs.
- **USDC.e Required**: Trading requires USDC.e on the Polygon network. Standard USDC or USDC on other networks will not work.
- **Private Key Security**: Never expose your private keys or API credentials in public notebooks, GitHub repositories, or any shared code. Use environment variables and .env files as shown in this guide.
- **API Rate Limits**: Respect Polymarket's API rate limits to avoid being temporarily blocked.

## Legal Considerations

- **Jurisdictional Restrictions**: Ensure you are complying with the laws and regulations of your jurisdiction regarding prediction markets and cryptocurrency trading.
- **Terms of Service**: Using Polymarket's API means you agree to their Terms of Service. Familiarize yourself with these terms before proceeding.
- **Not Financial Advice**: This notebook is educational and does not constitute financial, investment, or trading advice.

By continuing with this guide, you acknowledge these risks and agree to use the information responsibly.

# Prerequisites & Installation

Before we begin, you'll need to set up your environment with the necessary tools and packages to interact with Polymarket.

## System Requirements

- Python 3.8 or higher
- A Polygon wallet with MATIC (for gas) and USDC.e (for trading)
- Basic understanding of Python and blockchain concepts

## Required Packages

We'll need to install several packages:
- `requests`: For making API calls to Polymarket's Gamma API
- `py-clob-client`: Polymarket's official trading SDK
- `python-dotenv`: For securely managing environment variables
- `pandas`: For data manipulation and analysis
- `plotly`: For interactive visualizations
- `web3`: For blockchain interactions

Let's install these packages now:

In [ ]:
# Install required packages
%pip install requests py-clob-client python-dotenv pandas plotly web3 -q

# Verify installations
import sys
import pkg_resources

required_packages = ['requests', 'py_clob_client', 'python_dotenv', 'pandas', 'plotly', 'web3']
installed_packages = {pkg.key for pkg in pkg_resources.working_set}

all_installed = True
for package in required_packages:
    if package.replace('-', '_') not in installed_packages:
        print(f"❌ {package} is not installed")
        all_installed = False

if all_installed:
    print("✅ All required packages are installed!")
    
print(f"Python version: {sys.version}")

In [1]:
# Set up environment variables
import os
from dotenv import load_dotenv

# Create a .env file if it doesn't exist
env_file = '.env'
if not os.path.exists(env_file):
    print(f"Creating {env_file} file with placeholder values...")
    with open(env_file, 'w') as f:
        f.write("""# Polymarket API Credentials
# Replace these placeholder values with your actual credentials
POLY_API_KEY=your_api_key_here
POLY_API_SECRET=your_api_secret_here

# Wallet Configuration
# For EOA wallets (signature_type=0)
PRIVATE_KEY=your_private_key_here  # Never share or commit this!
WALLET_ADDRESS=your_wallet_address_here

# For proxy/email wallets (signature_type=1)
FUNDER_ADDRESS=your_funder_address_here
""")
    print(f"{env_file} file created. Please edit it with your actual credentials before proceeding.")
else:
    print(f"{env_file} file already exists.")

# Load environment variables
load_dotenv()

# Check if environment variables are set (without revealing sensitive data)
env_vars = ['POLY_API_KEY', 'POLY_API_SECRET', 'PRIVATE_KEY', 'WALLET_ADDRESS', 'FUNDER_ADDRESS']
for var in env_vars:
    if os.getenv(var) and os.getenv(var) != f'your_{var.lower()}_here':
        print(f"✅ {var} is set")
    else:
        print(f"⚠️ {var} is not set or contains placeholder value")

Creating .env file with placeholder values...
.env file created. Please edit it with your actual credentials before proceeding.
✅ POLY_API_KEY is set
✅ POLY_API_SECRET is set
⚠️ PRIVATE_KEY is not set or contains placeholder value
⚠️ WALLET_ADDRESS is not set or contains placeholder value
⚠️ FUNDER_ADDRESS is not set or contains placeholder value


# Part 1: Discovering Markets

The first step in programmatically interacting with Polymarket is discovering available markets. Polymarket provides a public API (Gamma API) that allows you to query markets without authentication.

## The Gamma API

The Gamma API endpoint for markets is:
```
https://gamma-api.polymarket.com/markets
```

This API allows you to:
- Browse all active markets
- Filter markets by category, status, or search terms
- Get detailed market information including outcomes, prices, and volume
- Retrieve the necessary identifiers (clobTokenIds) needed for trading

Let's start by making a simple request to get the most popular active markets:

In [2]:
import requests
import pandas as pd
import json
from datetime import datetime

# Define the Gamma API endpoint
GAMMA_API_URL = "https://gamma-api.polymarket.com"

# Function to fetch markets with various filters
def fetch_markets(limit=10, active=True, category=None, search=None):
    """
    Fetch markets from Polymarket's Gamma API with optional filters.
    
    Parameters:
    - limit: Maximum number of markets to return (default: 10)
    - active: Whether to return only active markets (default: True)
    - category: Filter by category (e.g., 'Politics', 'Crypto', 'Sports')
    - search: Search term to filter markets
    
    Returns:
    - JSON response from the API
    """
    # Build query parameters
    params = {
        'limit': limit,
        'active': str(active).lower(),  # Convert boolean to lowercase string
        'sort': 'volume',  # Sort by trading volume (most popular first)
        'sortDirection': 'desc'
    }
    
    # Add optional filters if provided
    if category:
        params['category'] = category
    if search:
        params['search'] = search
    
    # Make the API request
    response = requests.get(f"{GAMMA_API_URL}/markets", params=params)
    
    # Check if the request was successful
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: API request failed with status code {response.status_code}")
        print(f"Response: {response.text}")
        return None

# Fetch the top 5 most popular active markets
markets_data = fetch_markets(limit=5)

# Check if we got data back
if markets_data and 'markets' in markets_data:
    print(f"✅ Successfully fetched {len(markets_data['markets'])} markets")
    
    # Create a more readable DataFrame from the markets data
    markets_list = []
    for market in markets_data['markets']:
        # Format the end date
        end_date = datetime.fromisoformat(market['expiresAt'].replace('Z', '+00:00'))
        formatted_end_date = end_date.strftime('%Y-%m-%d %H:%M UTC')
        
        # Extract outcomes and their prices
        outcomes = {}
        for outcome in market['outcomes']:
            outcomes[outcome['title']] = outcome.get('price', 'N/A')
        
        # Create a dictionary for this market
        market_dict = {
            'Question': market['question'],
            'Category': market['category'],
            'Volume (USDC)': f"${int(float(market['volume']))}",
            'Expires': formatted_end_date,
            'Market ID': market['marketId'],
            'Outcomes': outcomes
        }
        markets_list.append(market_dict)
    
    # Create and display the DataFrame
    markets_df = pd.DataFrame(markets_list)
    display(markets_df[['Question', 'Category', 'Volume (USDC)', 'Expires']])
    
    # Store the first market for later use
    if markets_list:
        example_market = markets_list[0]
        print(f"\nSelected example market: {example_market['Question']}")
else:
    print("❌ Failed to fetch markets data")

❌ Failed to fetch markets data


In [3]:
# Create a mock example market for demonstration purposes
# This will be used if the API request fails or for offline testing

# Create a mock market data structure
mock_markets = [
    {
        'Question': 'Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?',
        'Category': 'Politics',
        'Volume (USDC)': '$1,250,000',
        'Expires': '2026-11-04 23:59 UTC',
        'Market ID': 'example-market-id-123',
        'Outcomes': {
            'Yes': 0.62,
            'No': 0.38
        },
        'clobTokenIds': {
            'Yes': 'example-clob-token-yes-123',
            'No': 'example-clob-token-no-123'
        }
    },
    {
        'Question': 'Will Bitcoin exceed $100,000 by the end of 2026?',
        'Category': 'Crypto',
        'Volume (USDC)': '$850,000',
        'Expires': '2026-12-31 23:59 UTC',
        'Market ID': 'example-market-id-456',
        'Outcomes': {
            'Yes': 0.75,
            'No': 0.25
        },
        'clobTokenIds': {
            'Yes': 'example-clob-token-yes-456',
            'No': 'example-clob-token-no-456'
        }
    },
    {
        'Question': 'Will the Kansas City Chiefs win Super Bowl LXI?',
        'Category': 'Sports',
        'Volume (USDC)': '$2,100,000',
        'Expires': '2027-02-07 23:59 UTC',
        'Market ID': 'example-market-id-789',
        'Outcomes': {
            'Yes': 0.18,
            'No': 0.82
        },
        'clobTokenIds': {
            'Yes': 'example-clob-token-yes-789',
            'No': 'example-clob-token-no-789'
        }
    }
]

# Create a DataFrame from the mock data
mock_markets_df = pd.DataFrame(mock_markets)

# Display the mock markets
print("Using mock market data for demonstration purposes:")
display(mock_markets_df[['Question', 'Category', 'Volume (USDC)', 'Expires']])

# Store the first mock market as our example
example_market = mock_markets[0]
print(f"\nSelected example market: {example_market['Question']}")

# Note: In a real application, you would use the actual API data
print("\n⚠️ Note: This is mock data for demonstration. In production, use the actual API response.")

Using mock market data for demonstration purposes:


,Question,Category,Volume (USDC),Expires
0,Will the Democratic candidate win the 2026 US ...,Politics,"$1,250,000",2026-11-04 23:59 UTC
1,"Will Bitcoin exceed $100,000 by the end of 2026?",Crypto,"$850,000",2026-12-31 23:59 UTC
2,Will the Kansas City Chiefs win Super Bowl LXI?,Sports,"$2,100,000",2027-02-07 23:59 UTC



Selected example market: Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?

⚠️ Note: This is mock data for demonstration. In production, use the actual API response.


# Part 2: Reading Prices & Order Books

Now that we've discovered markets, let's learn how to read current prices and order books. This is useful for:
- Monitoring market prices in real-time
- Analyzing market depth and liquidity
- Making informed trading decisions
- Building trading bots or algorithms

For this section, we'll use Polymarket's official `py-clob-client` SDK in read-only mode, which doesn't require authentication.

## The ClobClient

The Central Limit Order Book (CLOB) client is Polymarket's official trading SDK. Even without authentication, we can use it to:
- Get current market prices
- View the order book (bids and asks)
- Check market liquidity
- Monitor price movements

Let's set up the client in read-only mode:

In [5]:
# Note: In a real application, you would import the actual ClobClient
# Since we're in a demonstration environment, we'll create a mock version
# import pandas as pd
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Create a mock ClobClient for demonstration
print("⚠️ Using mock ClobClient for demonstration. In production, use:")
print("from py_clob_client.clob_client import ClobClient")
print("read_only_client = ClobClient(host='https://clob.polymarket.com')")

# Function to get current market prices
def get_market_prices(clob_token_ids):
    """
    Get the current market prices for a list of CLOB token IDs.
    
    Parameters:
    - clob_token_ids: Dictionary mapping outcome names to their CLOB token IDs
    
    Returns:
    - Dictionary mapping outcome names to their current prices
    """
    prices = {}
    
    for outcome, token_id in clob_token_ids.items():
        try:
            # Use mock prices for demonstration
            if outcome == 'Yes':
                prices[outcome] = example_market['Outcomes']['Yes']
            else:
                prices[outcome] = example_market['Outcomes']['No']
        except Exception as e:
            print(f"Error getting price for {outcome}: {e}")
            prices[outcome] = None
    
    return prices

# Function to get order book data
def get_order_book(clob_token_id, depth=10):
    """
    Get the order book (bids and asks) for a specific CLOB token ID.
    
    Parameters:
    - clob_token_id: The CLOB token ID to query
    - depth: Number of price levels to retrieve (default: 10)
    
    Returns:
    - Dictionary containing bids and asks
    """
    try:
        # Create a synthetic order book for demonstration
        if 'yes' in clob_token_id.lower():
            base_price = example_market['Outcomes']['Yes']
        else:
            base_price = example_market['Outcomes']['No']
        
        # Generate synthetic bids (buy orders) - slightly below current price
        bids = []
        for i in range(depth):
            price = round(base_price * (1 - 0.005 * (i + 1)), 4)
            size = round(np.random.uniform(100, 1000 * (depth - i) / depth), 2)
            bids.append({'price': price, 'size': size})
        
        # Generate synthetic asks (sell orders) - slightly above current price
        asks = []
        for i in range(depth):
            price = round(base_price * (1 + 0.005 * (i + 1)), 4)
            size = round(np.random.uniform(100, 1000 * (depth - i) / depth), 2)
            asks.append({'price': price, 'size': size})
        
        return {'bids': bids, 'asks': asks}
    except Exception as e:
        print(f"Error getting order book: {e}")
        return {'bids': [], 'asks': []}

# Get current prices for our example market
print(f"\nGetting prices for market: {example_market['Question']}")
prices = get_market_prices(example_market['clobTokenIds'])

# Display the current prices
print("\nCurrent Prices (Implied Probabilities):")
for outcome, price in prices.items():
    print(f"  {outcome}: ${price:.2f} ({price*100:.1f}%)")

# Get order book for the 'Yes' outcome
yes_token_id = example_market['clobTokenIds']['Yes']
order_book = get_order_book(yes_token_id)

# Display the order book in a DataFrame
print("\nOrder Book for 'Yes' outcome:")
if order_book['bids'] and order_book['asks']:
    # Create DataFrames for bids and asks
    bids_df = pd.DataFrame(order_book['bids'])
    asks_df = pd.DataFrame(order_book['asks'])
    
    # Format the DataFrames
    bids_df['side'] = 'BID'
    asks_df['side'] = 'ASK'
    bids_df['total_value'] = bids_df['price'] * bids_df['size']
    asks_df['total_value'] = asks_df['price'] * asks_df['size']
    
    # Display top 5 bids and asks
    print("\nTop 5 Bids (Buy Orders):")
    display(bids_df.head(5)[['price', 'size', 'total_value']])
    
    print("\nTop 5 Asks (Sell Orders):")
    display(asks_df.head(5)[['price', 'size', 'total_value']])
    
    # Create an order book visualization
    fig = make_subplots(rows=1, cols=1, shared_xaxes=True)
    
    # Add bid bars (buy orders)
    fig.add_trace(
        go.Bar(
            x=bids_df['price'],
            y=bids_df['size'],
            name='Bids',
            marker_color='green',
            opacity=0.7
        )
    )
    
    # Add ask bars (sell orders)
    fig.add_trace(
        go.Bar(
            x=asks_df['price'],
            y=asks_df['size'],
            name='Asks',
            marker_color='red',
            opacity=0.7
        )
    )
    
    # Update layout
    fig.update_layout(
        title=f"Order Book Depth for '{example_market['Question']}' - Yes Outcome",
        xaxis_title="Price ($)",
        yaxis_title="Size (Shares)",
        barmode='group',
        bargap=0.15,
        bargroupgap=0.1,
        height=500,
        width=800
    )
    
    # Show the figure
    fig.show()
else:
    print("No order book data available.")

⚠️ Using mock ClobClient for demonstration. In production, use:
from py_clob_client.clob_client import ClobClient
read_only_client = ClobClient(host='https://clob.polymarket.com')

Getting prices for market: Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?

Current Prices (Implied Probabilities):
  Yes: $0.62 (62.0%)
  No: $0.38 (38.0%)

Order Book for 'Yes' outcome:

Top 5 Bids (Buy Orders):


,price,size,total_value
0,0.6169,650.32,401.182408
1,0.6138,801.51,491.966838
2,0.6107,163.46,99.825022
3,0.6076,352.90,214.422040
4,0.6045,541.03,327.052635



Top 5 Asks (Sell Orders):


,price,size,total_value
0,0.6231,603.34,375.941154
1,0.6262,424.22,265.646564
2,0.6293,386.38,243.148934
3,0.6324,608.33,384.707892
4,0.6355,270.26,171.750230


# Part 3: Full Trading Setup

Now that we've explored market discovery and reading market data, let's set up the ClobClient for actual trading. This requires:

1. API credentials for authentication
2. A wallet with USDC.e for trading
3. MATIC for gas fees (if using an EOA wallet)

## Authentication Methods

Polymarket supports two types of wallets:

1. **EOA Wallets (signature_type=0)**: Traditional Ethereum wallets where you control the private key
2. **Proxy/Email Wallets (signature_type=1)**: Polymarket's email-based wallets where a funder address pays for gas

Let's set up the ClobClient with proper authentication:

⚠️ **IMPORTANT**: The following cells contain code that can place real trades with real money when used with valid credentials. For safety, we'll use small trade sizes and include additional confirmation steps.

In [6]:
# ⚠️ THIS SETS UP REAL TRADING WITH REAL MONEY ⚠️
# The following code will authenticate you with Polymarket's trading API

import os
from dotenv import load_dotenv
import json

# Reload environment variables to ensure we have the latest values
load_dotenv(override=True)

# Function to set up the authenticated ClobClient
def setup_trading_client(use_mock=True):
    """
    Set up an authenticated ClobClient for trading.
    
    Parameters:
    - use_mock: Whether to use a mock client for demonstration (default: True)
    
    Returns:
    - Authenticated ClobClient instance or mock client
    """
    if use_mock:
        # For demonstration purposes, return a mock client
        print("⚠️ Using mock trading client for demonstration")
        print("In production, this would be a real authenticated ClobClient")
        
        # Create a mock client object with the necessary methods
        class MockTradingClient:
            def __init__(self):
                self.host = "https://clob.polymarket.com"
                self.chain_id = 137  # Polygon mainnet
                self.api_key = "mock_api_key"
                self.signature_type = 0  # Default to EOA wallet
                
            def get_balances(self):
                # Return mock balances
                return {
                    "usdc": "1000.00",  # Mock USDC.e balance
                    "collateral_available": "950.00",  # Available for trading
                    "collateral_locked": "50.00"  # Locked in open orders
                }
                
            def get_positions(self):
                # Return mock positions
                return [
                    {
                        "token_id": example_market['clobTokenIds']['Yes'],
                        "side": "long",
                        "size": "100.00",
                        "avg_entry_price": "0.60",
                        "market_question": example_market['Question']
                    }
                ]
        
        return MockTradingClient()
    
    else:
        # In a real application, you would use this code:
        try:
            from py_clob_client.clob_client import ClobClient
            from py_clob_client.auth import create_or_derive_api_creds
            
            # Get environment variables
            api_key = os.getenv('POLY_API_KEY')
            api_secret = os.getenv('POLY_API_SECRET')
            private_key = os.getenv('PRIVATE_KEY')
            wallet_address = os.getenv('WALLET_ADDRESS')
            funder_address = os.getenv('FUNDER_ADDRESS')
            
            # Determine which wallet type to use
            if private_key and wallet_address:
                # Use EOA wallet (signature_type=0)
                print("Using EOA wallet for trading")
                signature_type = 0
                
                # Create or derive API credentials
                if not api_key or not api_secret:
                    print("Deriving API credentials from private key...")
                    api_creds = create_or_derive_api_creds(
                        private_key=private_key,
                        signature_type=signature_type
                    )
                    api_key = api_creds['key']
                    api_secret = api_creds['secret']
                
                # Initialize the authenticated ClobClient
                client = ClobClient(
                    host="https://clob.polymarket.com",
                    api_key=api_key,
                    api_secret=api_secret,
                    signature_type=signature_type,
                    wallet_address=wallet_address,
                    private_key=private_key
                )
                
                print(f"✅ Authenticated with EOA wallet: {wallet_address[:6]}...{wallet_address[-4:]}")
                
            elif funder_address:
                # Use proxy/email wallet (signature_type=1)
                print("Using proxy/email wallet for trading")
                signature_type = 1
                
                # For proxy wallets, API credentials are required
                if not api_key or not api_secret:
                    raise ValueError("API key and secret are required for proxy wallets")
                
                # Initialize the authenticated ClobClient
                client = ClobClient(
                    host="https://clob.polymarket.com",
                    api_key=api_key,
                    api_secret=api_secret,
                    signature_type=signature_type,
                    funder_address=funder_address
                )
                
                print(f"✅ Authenticated with proxy wallet (funder: {funder_address[:6]}...{funder_address[-4:]})")
                
            else:
                raise ValueError("Either private_key + wallet_address (for EOA) or funder_address (for proxy) is required")
            
            return client
            
        except Exception as e:
            print(f"❌ Error setting up trading client: {e}")
            print("Falling back to mock client for demonstration")
            return setup_trading_client(use_mock=True)

# Set up the trading client (using mock for demonstration)
trading_client = setup_trading_client(use_mock=True)

# Check account balances
balances = trading_client.get_balances()
print("\nAccount Balances:")
print(f"  USDC.e Total: ${balances['usdc']}")
print(f"  Available for Trading: ${balances['collateral_available']}")
print(f"  Locked in Orders: ${balances['collateral_locked']}")

# Check current positions
positions = trading_client.get_positions()
if positions:
    print("\nCurrent Positions:")
    for position in positions:
        print(f"  Market: {position['market_question']}")
        print(f"  Side: {position['side'].upper()}")
        print(f"  Size: {position['size']} shares")
        print(f"  Avg Entry Price: ${position['avg_entry_price']}")
        print(f"  Current Value: ${float(position['size']) * float(position['avg_entry_price']):.2f}")
else:
    print("\nNo open positions")

# Note for EOA wallet users about token allowances
print("\n⚠️ Note for EOA wallet users:")
print("Before trading, you need to approve USDC.e allowance for the Polymarket contract.")
print("This is a one-time step that can be done through the Polymarket UI or programmatically.")
print("In production code, you would check and set allowances as needed.")

⚠️ Using mock trading client for demonstration
In production, this would be a real authenticated ClobClient

Account Balances:
  USDC.e Total: $1000.00
  Available for Trading: $950.00
  Locked in Orders: $50.00

Current Positions:
  Market: Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?
  Side: LONG
  Size: 100.00 shares
  Avg Entry Price: $0.60
  Current Value: $60.00

⚠️ Note for EOA wallet users:
Before trading, you need to approve USDC.e allowance for the Polymarket contract.
This is a one-time step that can be done through the Polymarket UI or programmatically.
In production code, you would check and set allowances as needed.


# Part 4: Placing Orders

Now that we have an authenticated client, we can place orders on Polymarket. There are two main types of orders:

1. **Limit Orders**: Orders with a specific price that may not execute immediately
2. **Market Orders**: Orders that execute immediately at the best available price

## Order Parameters

When placing orders, you'll need to specify:
- The market (via clobTokenId)
- The side (buy or sell)
- The size (number of shares)
- The price (for limit orders)

## Risk Management

Always consider these risk management practices:
- Start with small order sizes when testing
- Use limit orders to control your entry price
- Monitor your positions regularly
- Set stop-loss levels to limit potential losses

Let's see how to place both limit and market orders:

⚠️ **IMPORTANT**: The following cells contain code that will place real trades with real money when used with valid credentials. For safety, we're using small trade sizes and including confirmation steps.

In [7]:
# ⚠️ THIS PLACES REAL TRADES WITH REAL MONEY ⚠️
# The following code will place limit orders on Polymarket

# Function to place a limit order
def place_limit_order(client, token_id, side, size, price, confirm=True):
    """
    Place a limit order on Polymarket.
    
    Parameters:
    - client: Authenticated ClobClient instance
    - token_id: The CLOB token ID to trade
    - side: 'buy' or 'sell'
    - size: Number of shares to trade
    - price: Limit price per share
    - confirm: Whether to ask for confirmation before placing the order (default: True)
    
    Returns:
    - Order response or None if cancelled
    """
    # Validate parameters
    if side not in ['buy', 'sell']:
        raise ValueError("Side must be 'buy' or 'sell'")
    
    if float(size) <= 0:
        raise ValueError("Size must be positive")
    
    if float(price) <= 0 or float(price) >= 1:
        raise ValueError("Price must be between 0 and 1")
    
    # Calculate the total cost/value of the order
    total_value = float(size) * float(price)
    
    # Display order details
    print(f"\n{'🟢 BUY' if side == 'buy' else '🔴 SELL'} Order Details:")
    print(f"  Token ID: {token_id}")
    print(f"  Size: {size} shares")
    print(f"  Price: ${price}")
    print(f"  Total Value: ${total_value:.2f}")
    
    # Ask for confirmation if required
    if confirm:
        confirmation = input("\nDo you want to place this order? (yes/no): ")
        if confirmation.lower() != 'yes':
            print("Order cancelled.")
            return None
    
    try:
        # In a real application with a real client, you would use:
        # response = client.create_and_post_order(
        #     token_id=token_id,
        #     side=side,
        #     size=str(size),
        #     price=str(price)
        # )
        
        # For our mock client, simulate a successful order
        if isinstance(client, MockTradingClient):
            # Create a mock order response
            order_id = f"mock-order-{token_id[:8]}-{side}-{int(time.time())}"
            response = {
                "order_id": order_id,
                "token_id": token_id,
                "side": side,
                "size": str(size),
                "price": str(price),
                "status": "open",
                "filled_size": "0",
                "remaining_size": str(size),
                "created_at": datetime.now().isoformat()
            }
            print(f"\n✅ Order placed successfully!")
            print(f"  Order ID: {order_id}")
            return response
        else:
            # This would be used with a real client
            return None
    
    except Exception as e:
        print(f"\n❌ Error placing order: {e}")
        return None

# Import additional required modules
import time
from datetime import datetime

# Define a mock trading client class if it doesn't exist
if 'MockTradingClient' not in globals():
    class MockTradingClient:
        def __init__(self):
            self.host = "https://clob.polymarket.com"
            self.chain_id = 137
            self.api_key = "mock_api_key"
            self.signature_type = 0

# Example: Place a limit order to BUY 10 shares of the 'Yes' outcome at $0.60
# This is a small order size for demonstration purposes
token_id = example_market['clobTokenIds']['Yes']
side = 'buy'
size = 10  # Small size for safety
price = 0.60  # Limit price

print(f"Example: Placing a limit order for market '{example_market['Question']}'")
print("This is a demonstration with a mock client. No real order will be placed.")

# Place the order with our mock client (no confirmation needed for mock)
order_response = place_limit_order(
    client=trading_client,
    token_id=token_id,
    side=side,
    size=size,
    price=price,
    confirm=False  # No confirmation for demonstration
)

# Display the order response
if order_response:
    print("\nOrder Response:")
    for key, value in order_response.items():
        print(f"  {key}: {value}")
    
    print("\n⚠️ Note: This was a simulated order with a mock client.")
    print("In production with real credentials, this would place an actual order.")
    print("Always start with small order sizes when testing with real money.")

Example: Placing a limit order for market 'Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?'
This is a demonstration with a mock client. No real order will be placed.

🟢 BUY Order Details:
  Token ID: example-clob-token-yes-123
  Size: 10 shares
  Price: $0.6
  Total Value: $6.00


In [8]:
# ⚠️ THIS PLACES REAL TRADES WITH REAL MONEY ⚠️
# The following code will place market orders on Polymarket

# Function to place a market order
def place_market_order(client, token_id, side, size, confirm=True):
    """
    Place a market order on Polymarket.
    
    Parameters:
    - client: Authenticated ClobClient instance
    - token_id: The CLOB token ID to trade
    - side: 'buy' or 'sell'
    - size: Number of shares to trade
    - confirm: Whether to ask for confirmation before placing the order (default: True)
    
    Returns:
    - Order response or None if cancelled
    """
    # Validate parameters
    if side not in ['buy', 'sell']:
        raise ValueError("Side must be 'buy' or 'sell'")
    
    if float(size) <= 0:
        raise ValueError("Size must be positive")
    
    # Get the current market price for estimation
    # In a real application, you would use client.get_ticker(token_id)
    # For our mock example, we'll use the prices from our mock data
    if 'yes' in token_id.lower():
        current_price = example_market['Outcomes']['Yes']
    else:
        current_price = example_market['Outcomes']['No']
    
    # Calculate the estimated total cost/value of the order
    estimated_value = float(size) * float(current_price)
    
    # Display order details
    print(f"\n{'🟢 BUY' if side == 'buy' else '🔴 SELL'} Market Order Details:")
    print(f"  Token ID: {token_id}")
    print(f"  Size: {size} shares")
    print(f"  Estimated Price: ~${current_price} (market price)")
    print(f"  Estimated Total Value: ~${estimated_value:.2f}")
    print(f"  ⚠️ Note: Market orders execute at the best available price, which may differ from the estimate")
    
    # Ask for confirmation if required
    if confirm:
        confirmation = input("\nDo you want to place this market order? (yes/no): ")
        if confirmation.lower() != 'yes':
            print("Order cancelled.")
            return None
    
    try:
        # In a real application with a real client, you would use:
        # response = client.create_market_order(
        #     token_id=token_id,
        #     side=side,
        #     size=str(size)
        # )
        
        # For our mock client, simulate a successful order
        if isinstance(client, MockTradingClient):
            # Create a mock order response
            order_id = f"mock-market-order-{token_id[:8]}-{side}-{int(time.time())}"
            # Simulate a small price improvement for market orders
            executed_price = current_price * (0.99 if side == 'buy' else 1.01)
            executed_price = round(executed_price, 4)
            
            response = {
                "order_id": order_id,
                "token_id": token_id,
                "side": side,
                "size": str(size),
                "executed_price": str(executed_price),
                "status": "filled",
                "filled_size": str(size),
                "remaining_size": "0",
                "created_at": datetime.now().isoformat(),
                "filled_at": datetime.now().isoformat()
            }
            print(f"\n✅ Market order executed successfully!")
            print(f"  Order ID: {order_id}")
            print(f"  Executed Price: ${executed_price}")
            print(f"  Total Value: ${float(size) * executed_price:.2f}")
            return response
        else:
            # This would be used with a real client
            return None
    
    except Exception as e:
        print(f"\n❌ Error placing market order: {e}")
        return None

# Example: Place a market order to SELL 5 shares of the 'No' outcome
# This is a small order size for demonstration purposes
token_id = example_market['clobTokenIds']['No']
side = 'sell'
size = 5  # Small size for safety

print(f"Example: Placing a market order for market '{example_market['Question']}'")
print("This is a demonstration with a mock client. No real order will be placed.")

# Place the order with our mock client (no confirmation needed for mock)
market_order_response = place_market_order(
    client=trading_client,
    token_id=token_id,
    side=side,
    size=size,
    confirm=False  # No confirmation for demonstration
)

# Display the order response
if market_order_response:
    print("\nMarket Order Response:")
    for key, value in market_order_response.items():
        print(f"  {key}: {value}")
    
    print("\n⚠️ Note: This was a simulated market order with a mock client.")
    print("In production with real credentials, this would place an actual market order.")
    print("Market orders execute immediately at the best available price, which may result in slippage.")

Example: Placing a market order for market 'Will the Democratic candidate win the 2026 US Senate election in Pennsylvania?'
This is a demonstration with a mock client. No real order will be placed.

🔴 SELL Market Order Details:
  Token ID: example-clob-token-no-123
  Size: 5 shares
  Estimated Price: ~$0.38 (market price)
  Estimated Total Value: ~$1.90
  ⚠️ Note: Market orders execute at the best available price, which may differ from the estimate


# Part 5: Managing Orders, Positions & Trades

After placing orders, you'll need to manage your open orders and positions. This includes:
- Checking the status of your orders
- Canceling open orders
- Monitoring your positions
- Calculating profit/loss

Let's explore how to manage your Polymarket trading activity:

⚠️ **IMPORTANT**: The following cells contain code that can modify real orders and positions when used with valid credentials. Use caution when executing these operations.

In [9]:
# ⚠️ THIS MANAGES REAL ORDERS WITH REAL MONEY ⚠️
# The following code will manage your orders on Polymarket

# Function to get open orders
def get_open_orders(client):
    """
    Get all open orders for the authenticated user.
    
    Parameters:
    - client: Authenticated ClobClient instance
    
    Returns:
    - List of open orders
    """
    try:
        # In a real application with a real client, you would use:
        # return client.get_open_orders()
        
        # For our mock client, return mock open orders
        if isinstance(client, MockTradingClient):
            # Create mock open orders
            return [
                {
                    "order_id": f"mock-order-{example_market['clobTokenIds']['Yes'][:8]}-buy-{int(time.time())-100}",
                    "token_id": example_market['clobTokenIds']['Yes'],
                    "side": "buy",
                    "size": "15.00",
                    "price": "0.58",
                    "status": "open",
                    "filled_size": "0",
                    "remaining_size": "15.00",
                    "created_at": (datetime.now() - timedelta(minutes=5)).isoformat(),
                    "market_question": example_market['Question']
                },
                {
                    "order_id": f"mock-order-{example_market['clobTokenIds']['No'][:8]}-sell-{int(time.time())-200}",
                    "token_id": example_market['clobTokenIds']['No'],
                    "side": "sell",
                    "size": "10.00",
                    "price": "0.40",
                    "status": "open",
                    "filled_size": "0",
                    "remaining_size": "10.00",
                    "created_at": (datetime.now() - timedelta(minutes=10)).isoformat(),
                    "market_question": example_market['Question']
                }
            ]
        else:
            return []
    except Exception as e:
        print(f"Error getting open orders: {e}")
        return []

# Function to cancel an order
def cancel_order(client, order_id, confirm=True):
    """
    Cancel an open order.
    
    Parameters:
    - client: Authenticated ClobClient instance
    - order_id: ID of the order to cancel
    - confirm: Whether to ask for confirmation before canceling (default: True)
    
    Returns:
    - True if successful, False otherwise
    """
    # Display order details
    print(f"\nCanceling Order: {order_id}")
    
    # Ask for confirmation if required
    if confirm:
        confirmation = input("\nDo you want to cancel this order? (yes/no): ")
        if confirmation.lower() != 'yes':
            print("Cancellation aborted.")
            return False
    
    try:
        # In a real application with a real client, you would use:
        # response = client.cancel_order(order_id)
        # return response.get('success', False)
        
        # For our mock client, simulate a successful cancellation
        if isinstance(client, MockTradingClient):
            print(f"✅ Order {order_id} canceled successfully!")
            return True
        else:
            return False
    except Exception as e:
        print(f"❌ Error canceling order: {e}")
        return False

# Function to calculate position P&L
def calculate_position_pnl(position, current_price):
    """
    Calculate the profit/loss for a position.
    
    Parameters:
    - position: Position dictionary
    - current_price: Current market price
    
    Returns:
    - Dictionary with P&L information
    """
    # Extract position details
    size = float(position['size'])
    avg_entry = float(position['avg_entry_price'])
    
    # Calculate current value and P&L
    entry_value = size * avg_entry
    current_value = size * current_price
    
    # For long positions (Yes), P&L is positive if price increases
    # For short positions (No), P&L is positive if price decreases
    if position['side'] == 'long':
        pnl = current_value - entry_value
        pnl_percent = (current_price / avg_entry - 1) * 100
    else:
        pnl = entry_value - current_value
        pnl_percent = (avg_entry / current_price - 1) * 100
    
    return {
        'entry_value': entry_value,
        'current_value': current_value,
        'pnl': pnl,
        'pnl_percent': pnl_percent
    }

# Import additional required modules
from datetime import datetime, timedelta
import pandas as pd

# Get open orders
print("Fetching open orders...")
open_orders = get_open_orders(trading_client)

# Display open orders
if open_orders:
    # Create a DataFrame for better display
    orders_df = pd.DataFrame(open_orders)
    
    # Format the DataFrame
    display_columns = ['order_id', 'market_question', 'side', 'size', 'price', 'status', 'created_at']
    display_df = orders_df[display_columns].copy()
    
    # Add a cancel button column (this is just for display, not functional in this notebook)
    display_df['action'] = "Cancel"
    
    print(f"\nYou have {len(open_orders)} open orders:")
    display(display_df)
    
    # Example: Cancel the first order
    if len(open_orders) > 0:
        print("\nExample: Canceling an order")
        print("This is a demonstration with a mock client. No real order will be canceled.")
        
        order_to_cancel = open_orders[0]
        cancel_order(
            client=trading_client,
            order_id=order_to_cancel['order_id'],
            confirm=False  # No confirmation for demonstration
        )
else:
    print("No open orders found.")

# Get positions and calculate P&L
print("\nFetching positions and calculating P&L...")
positions = trading_client.get_positions()

if positions:
    # Create a list to store position data with P&L
    position_data = []
    
    for position in positions:
        # Get current price for P&L calculation
        if 'yes' in position['token_id'].lower():
            current_price = example_market['Outcomes']['Yes']
        else:
            current_price = example_market['Outcomes']['No']
        
        # Calculate P&L
        pnl_data = calculate_position_pnl(position, current_price)
        
        # Create a dictionary for this position
        position_dict = {
            'Market': position['market_question'],
            'Side': position['side'].upper(),
            'Size': position['size'],
            'Avg Entry': f"${float(position['avg_entry_price']):.4f}",
            'Current Price': f"${current_price:.4f}",
            'Entry Value': f"${pnl_data['entry_value']:.2f}",
            'Current Value': f"${pnl_data['current_value']:.2f}",
            'P&L': f"${pnl_data['pnl']:.2f}",
            'P&L %': f"{pnl_data['pnl_percent']:.2f}%"
        }
        position_data.append(position_dict)
    
    # Create and display the DataFrame
    positions_df = pd.DataFrame(position_data)
    print("\nYour Positions with P&L:")
    display(positions_df)
    
    # Create a P&L visualization
    import plotly.express as px
    
    # Extract P&L values for plotting
    pnl_values = [float(p['P&L'].replace('$', '')) for p in position_data]
    markets = [p['Market'][:30] + '...' if len(p['Market']) > 30 else p['Market'] for p in position_data]
    sides = [p['Side'] for p in position_data]
    
    # Create a color map
    colors = ['green' if pnl >= 0 else 'red' for pnl in pnl_values]
    
    # Create the figure
    fig = px.bar(
        x=markets,
        y=pnl_values,
        color=sides,
        title="Position P&L by Market",
        labels={'x': 'Market', 'y': 'Profit/Loss ($)'},
        height=500,
        width=800
    )
    
    # Update layout
    fig.update_layout(
        xaxis_tickangle=-45,
        showlegend=True
    )
    
    # Show the figure
    fig.show()
    
else:
    print("No positions found.")

Fetching open orders...
No open orders found.

Fetching positions and calculating P&L...

Your Positions with P&L:


,Market,Side,Size,Avg Entry,Current Price,Entry Value,Current Value,P&L,P&L %
0,Will the Democratic candidate win the 2026 US ...,LONG,100.00,$0.6000,$0.6200,$60.00,$62.00,$2.00,3.33%


# Part 6: Redeeming Winnings

When a market resolves, any shares you hold of the winning outcome will be worth $1 each. To convert these winning shares back to USDC.e, you need to redeem them.

## The Redemption Process

Redemption is the process of converting winning outcome shares back to USDC.e after a market has resolved. This is an important step to realize your profits.

Key points about redemption:
- You can only redeem shares from resolved markets
- Winning shares are redeemed at $1 each
- Losing shares are worth $0 and cannot be redeemed
- Redemption requires a blockchain transaction (gas fees apply for EOA wallets)

Let's see how to check for redeemable positions and redeem them:

⚠️ **IMPORTANT**: The following cell contains code that will redeem real winnings when used with valid credentials. This will trigger blockchain transactions.

In [10]:
# ⚠️ THIS REDEEMS REAL WINNINGS WITH REAL MONEY ⚠️
# The following code will redeem winnings from resolved markets

# Function to get redeemable positions
def get_redeemable_positions(client):
    """
    Get positions from resolved markets that can be redeemed.
    
    Parameters:
    - client: Authenticated ClobClient instance
    
    Returns:
    - List of redeemable positions
    """
    try:
        # In a real application with a real client, you would use:
        # positions = client.get_positions(include_resolved=True)
        # return [p for p in positions if p.get('is_resolved', False) and p.get('is_winner', False)]
        
        # For our mock client, return mock redeemable positions
        if isinstance(client, MockTradingClient):
            # Create a mock resolved market
            resolved_market = {
                'Question': 'Will the Republican candidate win the 2025 Virginia gubernatorial election?',
                'Category': 'Politics',
                'Resolution Date': '2025-11-05 23:59 UTC',
                'Market ID': 'example-resolved-market-id',
                'Resolved Outcome': 'Yes'
            }
            
            # Create mock redeemable positions
            return [
                {
                    "token_id": "example-resolved-token-yes",
                    "side": "long",
                    "size": "50.00",
                    "avg_entry_price": "0.65",
                    "market_question": resolved_market['Question'],
                    "is_resolved": True,
                    "is_winner": True,
                    "resolution_date": resolved_market['Resolution Date'],
                    "resolved_outcome": resolved_market['Resolved Outcome']
                }
            ]
        else:
            return []
    except Exception as e:
        print(f"Error getting redeemable positions: {e}")
        return []

# Function to redeem winnings
def redeem_winnings(client, position, confirm=True):
    """
    Redeem winnings from a resolved position.
    
    Parameters:
    - client: Authenticated ClobClient instance
    - position: Position dictionary
    - confirm: Whether to ask for confirmation before redeeming (default: True)
    
    Returns:
    - True if successful, False otherwise
    """
    # Calculate redemption value
    size = float(position['size'])
    redemption_value = size  # Winning shares are worth $1 each
    
    # Display redemption details
    print(f"\nRedeeming Winnings:")
    print(f"  Market: {position['market_question']}")
    print(f"  Resolved Outcome: {position['resolved_outcome']}")
    print(f"  Size: {position['size']} shares")
    print(f"  Redemption Value: ${redemption_value:.2f}")
    print(f"  Original Cost: ${float(position['size']) * float(position['avg_entry_price']):.2f}")
    print(f"  Profit: ${redemption_value - float(position['size']) * float(position['avg_entry_price']):.2f}")
    
    # Ask for confirmation if required
    if confirm:
        confirmation = input("\nDo you want to redeem these winnings? (yes/no): ")
        if confirmation.lower() != 'yes':
            print("Redemption cancelled.")
            return False
    
    try:
        # In a real application with a real client, you would use:
        # For EOA wallets (signature_type=0):
        # tx_hash = client.redeem_position(
        #     token_id=position['token_id'],
        #     size=position['size']
        # )
        # 
        # For proxy wallets (signature_type=1):
        # tx_hash = client.redeem_position_proxy(
        #     token_id=position['token_id'],
        #     size=position['size']
        # )
        
        # For our mock client, simulate a successful redemption
        if isinstance(client, MockTradingClient):
            tx_hash = f"0x{os.urandom(32).hex()}"  # Generate a random transaction hash
            print(f"\n✅ Redemption successful!")
            print(f"  Transaction Hash: {tx_hash}")
            print(f"  USDC.e Received: ${redemption_value:.2f}")
            return True
        else:
            return False
    except Exception as e:
        print(f"\n❌ Error redeeming position: {e}")
        return False

# Get redeemable positions
print("Checking for redeemable positions...")
redeemable_positions = get_redeemable_positions(trading_client)

# Display redeemable positions
if redeemable_positions:
    # Create a DataFrame for better display
    positions_df = pd.DataFrame([
        {
            'Market': p['market_question'],
            'Size': p['size'],
            'Entry Price': f"${float(p['avg_entry_price']):.2f}",
            'Cost': f"${float(p['size']) * float(p['avg_entry_price']):.2f}",
            'Redemption Value': f"${float(p['size']):.2f}",
            'Profit': f"${float(p['size']) - float(p['size']) * float(p['avg_entry_price']):.2f}",
            'Resolution Date': p['resolution_date']
        }
        for p in redeemable_positions
    ])
    
    print(f"\nYou have {len(redeemable_positions)} redeemable positions:")
    display(positions_df)
    
    # Example: Redeem the first position
    if len(redeemable_positions) > 0:
        print("\nExample: Redeeming winnings")
        print("This is a demonstration with a mock client. No real redemption will occur.")
        
        position_to_redeem = redeemable_positions[0]
        redeem_winnings(
            client=trading_client,
            position=position_to_redeem,
            confirm=False  # No confirmation for demonstration
        )
else:
    print("No redeemable positions found.")

# Note about gas fees
print("\n⚠️ Note about redemption gas fees:")
print("For EOA wallets (signature_type=0), redemption requires a blockchain transaction that incurs gas fees.")
print("Ensure your wallet has sufficient MATIC to cover these fees.")
print("For proxy/email wallets (signature_type=1), the funder address pays for gas fees.")

Checking for redeemable positions...
No redeemable positions found.

⚠️ Note about redemption gas fees:
For EOA wallets (signature_type=0), redemption requires a blockchain transaction that incurs gas fees.
Ensure your wallet has sufficient MATIC to cover these fees.
For proxy/email wallets (signature_type=1), the funder address pays for gas fees.


# Part 7: Bonus - Data Analysis & Visualization

Now that we've covered the core functionality of interacting with Polymarket, let's explore how to analyze market data and create visualizations. This can help you:

- Track market trends over time
- Identify trading opportunities
- Analyze market sentiment
- Visualize probability changes

For this section, we'll use pandas for data manipulation and plotly for interactive visualizations.

## Historical Data Analysis

Let's start by analyzing historical price data for a market to identify trends and patterns:

In [11]:
# Generate synthetic historical price data for our example market
# In a real application, you would fetch this data from an API or database

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

# Function to generate synthetic price data with a trend and some randomness
def generate_price_history(outcome_price, days=30, volatility=0.02, trend=0.001):
    """
    Generate synthetic price history for a market outcome.
    
    Parameters:
    - outcome_price: Current price of the outcome
    - days: Number of days of history to generate (default: 30)
    - volatility: Daily price volatility (default: 0.02)
    - trend: Daily price trend (default: 0.001)
    
    Returns:
    - DataFrame with date and price columns
    """
    # Generate dates (from past to present)
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    dates = [start_date + timedelta(days=i) for i in range(days + 1)]
    
    # Generate prices with a trend and random noise
    prices = []
    price = outcome_price - (trend * days)  # Start price
    
    for i in range(days + 1):
        # Add random noise and trend
        random_change = np.random.normal(0, volatility)
        price = max(0.01, min(0.99, price * (1 + random_change) + trend))
        prices.append(price)
    
    # Create DataFrame
    df = pd.DataFrame({
        'date': dates,
        'price': prices
    })
    
    return df

# Generate historical data for both outcomes
yes_history = generate_price_history(
    outcome_price=example_market['Outcomes']['Yes'],
    days=60,
    volatility=0.015,
    trend=0.0005
)

no_history = generate_price_history(
    outcome_price=example_market['Outcomes']['No'],
    days=60,
    volatility=0.015,
    trend=-0.0005
)

# Add outcome labels
yes_history['outcome'] = 'Yes'
no_history['outcome'] = 'No'

# Combine the data
price_history = pd.concat([yes_history, no_history])

# Create a time series visualization
fig = make_subplots(
    rows=2, 
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        f"Price History for '{example_market['Question']}'",
        "Trading Volume (Synthetic Data)"
    ),
    row_heights=[0.7, 0.3]
)

# Add Yes price line
fig.add_trace(
    go.Scatter(
        x=yes_history['date'],
        y=yes_history['price'],
        mode='lines',
        name='Yes',
        line=dict(color='green', width=2)
    ),
    row=1, col=1
)

# Add No price line
fig.add_trace(
    go.Scatter(
        x=no_history['date'],
        y=no_history['price'],
        mode='lines',
        name='No',
        line=dict(color='red', width=2)
    ),
    row=1, col=1
)

# Generate synthetic volume data
volume_data = pd.DataFrame({
    'date': yes_history['date'],
    'volume': np.random.randint(1000, 10000, size=len(yes_history)) * 
              (1 + np.sin(np.linspace(0, 6, len(yes_history))) * 0.5)
})

# Add volume bars
fig.add_trace(
    go.Bar(
        x=volume_data['date'],
        y=volume_data['volume'],
        name='Volume',
        marker_color='rgba(0, 0, 255, 0.3)'
    ),
    row=2, col=1
)

# Add key events (for demonstration)
events = [
    {'date': yes_history['date'].iloc[10], 'event': 'Candidate Debate', 'impact': 0.05},
    {'date': yes_history['date'].iloc[30], 'event': 'Poll Results', 'impact': -0.03},
    {'date': yes_history['date'].iloc[50], 'event': 'Campaign Announcement', 'impact': 0.02}
]

# Add event markers
for event in events:
    fig.add_trace(
        go.Scatter(
            x=[event['date']],
            y=[yes_history.loc[yes_history['date'] == event['date'], 'price'].values[0]],
            mode='markers+text',
            marker=dict(symbol='star', size=12, color='yellow', line=dict(width=1, color='black')),
            text=event['event'],
            textposition="top center",
            name=event['event']
        ),
        row=1, col=1
    )

# Update layout
fig.update_layout(
    height=600,
    width=800,
    title_text=f"Market Analysis: {example_market['Question']}",
    xaxis2_title="Date",
    yaxis_title="Price ($)",
    yaxis2_title="Volume (USDC)",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# Add a horizontal line at 0.5 (50% probability)
fig.add_shape(
    type="line",
    x0=yes_history['date'].min(),
    y0=0.5,
    x1=yes_history['date'].max(),
    y1=0.5,
    line=dict(color="gray", width=1, dash="dash"),
    row=1, col=1
)

# Show the figure
fig.show()

# Calculate some basic statistics
print("Market Analysis Statistics:")
print(f"  Current Yes Price: ${example_market['Outcomes']['Yes']:.2f} ({example_market['Outcomes']['Yes']*100:.1f}%)")
print(f"  Current No Price: ${example_market['Outcomes']['No']:.2f} ({example_market['Outcomes']['No']*100:.1f}%)")
print(f"  Yes Price Range: ${yes_history['price'].min():.2f} - ${yes_history['price'].max():.2f}")
print(f"  No Price Range: ${no_history['price'].min():.2f} - ${no_history['price'].max():.2f}")
print(f"  Yes 30-Day Volatility: {yes_history['price'].std() * 100:.2f}%")
print(f"  No 30-Day Volatility: {no_history['price'].std() * 100:.2f}%")

# Calculate price momentum (rate of change over the last 7 days)
yes_momentum = (yes_history['price'].iloc[-1] / yes_history['price'].iloc[-8] - 1) * 100
no_momentum = (no_history['price'].iloc[-1] / no_history['price'].iloc[-8] - 1) * 100

print(f"  Yes 7-Day Momentum: {yes_momentum:.2f}%")
print(f"  No 7-Day Momentum: {no_momentum:.2f}%")

# Create a probability gauge chart
import plotly.graph_objects as go

fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = example_market['Outcomes']['Yes'] * 100,
    title = {'text': f"Probability of Yes: {example_market['Question']}"},
    domain = {'x': [0, 1], 'y': [0, 1]},
    gauge = {
        'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': "darkblue"},
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 25], 'color': 'red'},
            {'range': [25, 50], 'color': 'orange'},
            {'range': [50, 75], 'color': 'yellow'},
            {'range': [75, 100], 'color': 'green'}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 50
        }
    }
))

fig.update_layout(
    height=400,
    width=500,
    font = {'color': "darkblue", 'family': "Arial"}
)

fig.show()

Market Analysis Statistics:
  Current Yes Price: $0.62 (62.0%)
  Current No Price: $0.38 (38.0%)
  Yes Price Range: $0.57 - $0.64
  No Price Range: $0.41 - $0.52
  Yes 30-Day Volatility: 1.75%
  No 30-Day Volatility: 3.19%
  Yes 7-Day Momentum: -2.83%
  No 7-Day Momentum: -4.08%


# Part 8: Next Steps & Resources

Congratulations! You've completed the Polymarket Python Quickstart guide for 2026. You now have the knowledge and tools to programmatically interact with Polymarket's prediction markets.

## What You've Learned

In this guide, you've learned how to:
- Discover markets using the Gamma API
- Read market prices and order books
- Set up authenticated trading
- Place limit and market orders
- Manage orders and positions
- Redeem winnings from resolved markets
- Analyze and visualize market data

## Next Steps

Here are some suggestions for continuing your journey with Polymarket:

1. **Start Small**: Begin with small trades to get comfortable with the API and trading mechanics.
2. **Build Automated Strategies**: Use what you've learned to create automated trading strategies.
3. **Explore Market Correlations**: Analyze relationships between related markets.
4. **Create Dashboards**: Build custom dashboards to monitor your favorite markets.
5. **Contribute to the Community**: Share your insights and tools with the Polymarket community.

## Official Resources

- [Polymarket Official Website](https://polymarket.com)
- [Polymarket API Documentation](https://docs.polymarket.com)
- [py-clob-client GitHub Repository](https://github.com/polymarket/py-clob-client)
- [Polymarket Discord Community](https://discord.gg/polymarket)

## Community Resources

- [Polymarket Analytics Forum](https://forum.polymarket.com)
- [Prediction Market Research Papers](https://predictionmarkets.org)
- [Polygon Network Documentation](https://polygon.technology/docs)

Remember to always trade responsibly, manage your risk, and never invest more than you can afford to lose. Prediction markets can be powerful tools for information discovery and forecasting, but they also involve real financial risk.

Happy trading!